# Column Operations
> `MXColumn` — a MAX-backed column expression with arithmetic, comparisons, and aggregations.
>
> Storage: PyArrow `ChunkedArray` (zero-copy, schema-aware).  
> Compute: `max.experimental.tensor` for element-wise ops; `MAXSession` for Graph-based aggregations.  
> Device: explicit `device='cpu'|'gpu'` throughout — no magic auto-dispatch.

In [ ]:
#| default_exp ops

In [ ]:
#| export
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
from typing import Union, Optional, Literal

from max import driver
from max.experimental import tensor as mx_tensor

from mxframe.core_bridge import MXFrame, arrow_to_numpy_view, DeviceType
from mxframe.max_engine import (
    MAXSession, max_sum, max_mean, max_min, max_max, max_count, max_masked_sum
)

Scalar = Union[int, float, bool]

## MXColumn

In [ ]:
#| export
class MXColumn:
    """
    A named, device-aware column backed by a PyArrow `ChunkedArray`.

    Arithmetic and comparison operators use `max.experimental.tensor` so the
    same expression runs on CPU or GPU depending on `device`.  Aggregations
    use the pre-compiled `MAXSession` Graph API.

    Examples:
        >>> c = MXColumn.from_list([1.0, 2.0, 3.0], name="x")
        >>> (c * 2 + 1).to_pylist()
        [3.0, 5.0, 7.0]
        >>> (c > 1).to_pylist()
        [False, True, True]
        >>> c.sum(device="cpu")
        6.0
    """

    # ------------------------------------------------------------------
    # Construction
    # ------------------------------------------------------------------

    def __init__(
        self,
        data:   Union[pa.ChunkedArray, pa.Array],
        name:   str        = "",
        device: DeviceType = "cpu",
    ):
        if isinstance(data, pa.Array):
            data = pa.chunked_array([data])
        if not isinstance(data, pa.ChunkedArray):
            raise TypeError(f"Expected ChunkedArray or Array, got {type(data)}")
        self._data   = data
        self._name   = name
        self._device = device

    @classmethod
    def from_list(cls, data: list, name: str = "", device: DeviceType = "cpu") -> 'MXColumn':
        """Create from a Python list."""
        return cls(pa.chunked_array([pa.array(data)]), name=name, device=device)

    @classmethod
    def from_numpy(cls, arr: np.ndarray, name: str = "", device: DeviceType = "cpu") -> 'MXColumn':
        """Create from a NumPy array."""
        return cls(pa.chunked_array([pa.array(arr)]), name=name, device=device)

    # ------------------------------------------------------------------
    # Properties
    # ------------------------------------------------------------------

    @property
    def name(self) -> str:   return self._name

    @property
    def dtype(self) -> pa.DataType: return self._data.type

    @property
    def device(self) -> DeviceType: return self._device

    def __len__(self) -> int: return len(self._data)

    # ------------------------------------------------------------------
    # Conversion
    # ------------------------------------------------------------------

    def to_arrow(self) -> pa.ChunkedArray:  return self._data

    def to_numpy(self) -> np.ndarray:
        return self._data.to_numpy(zero_copy_only=False)

    def to_pylist(self) -> list: return self._data.to_pylist()

    def to(self, device: DeviceType) -> 'MXColumn':
        """Return a view of this column targeting a different device."""
        return MXColumn(self._data, self._name, device)

    # ------------------------------------------------------------------
    # Internal: numpy → MAX experimental tensor → result → MXColumn
    # ------------------------------------------------------------------

    def _mx_tensor(self) -> mx_tensor.Tensor:
        arr = np.ascontiguousarray(self.to_numpy(), dtype=np.float32)
        t   = mx_tensor.Tensor.from_dlpack(arr)
        if self._device == "gpu":
            if driver.accelerator_count() == 0:
                raise RuntimeError("No GPU available. Use device='cpu'.")
            t = t.to(driver.Accelerator())
        return t

    @classmethod
    def _from_mx(cls, t: mx_tensor.Tensor, name: str = "", device: DeviceType = "cpu") -> 'MXColumn':
        """MAX experimental tensor → MXColumn (D2H if on GPU)."""
        arr = np.from_dlpack(t.to(driver.CPU()))
        return cls.from_numpy(arr, name=name, device=device)

    def _other_mx(self, other: Union['MXColumn', Scalar]) -> mx_tensor.Tensor:
        """Coerce `other` to an mx tensor on the same device."""
        if isinstance(other, MXColumn):
            return other._mx_tensor()
        # Scalar — broadcast via numpy
        arr = np.full(len(self), float(other), dtype=np.float32)
        t   = mx_tensor.Tensor.from_dlpack(arr)
        if self._device == "gpu":
            t = t.to(driver.Accelerator())
        return t

    def _binop(self, op: str, other: Union['MXColumn', Scalar]) -> 'MXColumn':
        """Apply a named binary op via MAX experimental tensor."""
        a, b = self._mx_tensor(), self._other_mx(other)
        result = getattr(a, op)(b) if hasattr(a, op) else _pa_binop(op, self, other)
        return MXColumn._from_mx(result, device=self._device)

    # ------------------------------------------------------------------
    # Arithmetic
    # ------------------------------------------------------------------

    def __add__(self, other):   return self._binop("__add__", other)
    def __radd__(self, other):  return MXColumn.from_numpy(np.full(len(self), float(other), dtype=np.float32))._binop("__add__", self)
    def __sub__(self, other):   return self._binop("__sub__", other)
    def __rsub__(self, other):  return MXColumn.from_numpy(np.full(len(self), float(other), dtype=np.float32))._binop("__sub__", self)
    def __mul__(self, other):   return self._binop("__mul__", other)
    def __rmul__(self, other):  return self._binop("__mul__", other)
    def __truediv__(self, other): return self._binop("__truediv__", other)
    def __rtruediv__(self, other):
        return MXColumn.from_numpy(np.full(len(self), float(other), dtype=np.float32))._binop("__truediv__", self)
    def __neg__(self): return MXColumn._from_mx(-self._mx_tensor(), device=self._device)

    # ------------------------------------------------------------------
    # Comparisons  (return boolean MXColumn)
    # ------------------------------------------------------------------

    def _cmp(self, op: str, other) -> 'MXColumn':
        a_np = self.to_numpy().astype(np.float32)
        if isinstance(other, MXColumn):
            b_np = other.to_numpy().astype(np.float32)
        else:
            b_np = np.full(len(self), float(other), dtype=np.float32)
        result = getattr(np, op)(a_np, b_np)   # bool array
        return MXColumn.from_numpy(result.astype(np.float32), device=self._device)

    def __gt__(self, other):  return self._cmp("greater",       other)
    def __ge__(self, other):  return self._cmp("greater_equal", other)
    def __lt__(self, other):  return self._cmp("less",          other)
    def __le__(self, other):  return self._cmp("less_equal",    other)
    def __eq__(self, other):  return self._cmp("equal",         other)
    def __ne__(self, other):  return self._cmp("not_equal",     other)

    # ------------------------------------------------------------------
    # Boolean logic
    # ------------------------------------------------------------------

    def __and__(self, other: 'MXColumn') -> 'MXColumn':
        arr = (self.to_numpy().astype(bool) & other.to_numpy().astype(bool)).astype(np.float32)
        return MXColumn.from_numpy(arr, device=self._device)

    def __or__(self, other: 'MXColumn') -> 'MXColumn':
        arr = (self.to_numpy().astype(bool) | other.to_numpy().astype(bool)).astype(np.float32)
        return MXColumn.from_numpy(arr, device=self._device)

    def __invert__(self) -> 'MXColumn':
        arr = (~self.to_numpy().astype(bool)).astype(np.float32)
        return MXColumn.from_numpy(arr, device=self._device)

    # ------------------------------------------------------------------
    # Aggregations (MAX Engine — Graph API)
    # ------------------------------------------------------------------

    def sum(self, device: DeviceType = None) -> float:
        """Sum via MAX Graph API (compiled once per shape)."""
        return max_sum(self.to_numpy(), device=device or self._device)

    def mean(self, device: DeviceType = None) -> float:
        """Mean via MAX experimental tensor."""
        return max_mean(self.to_numpy(), device=device or self._device)

    def min(self, device: DeviceType = None) -> float:
        """Min via MAX experimental tensor."""
        return max_min(self.to_numpy(), device=device or self._device)

    def max(self, device: DeviceType = None) -> float:
        """Max via MAX experimental tensor."""
        return max_max(self.to_numpy(), device=device or self._device)

    def count(self) -> int:
        """Element count."""
        return max_count(self.to_numpy())

    # ------------------------------------------------------------------
    # Display
    # ------------------------------------------------------------------

    def __repr__(self) -> str:
        name = f"'{self._name}' " if self._name else ""
        return f"MXColumn {name}[{len(self)} × {self.dtype}, device={self._device}]"

## PyArrow compute fallback
Used for operations not yet exposed by `max.experimental.tensor`.

In [ ]:
#| export
def _pa_binop(op: str, a: MXColumn, b: Union[MXColumn, Scalar]) -> MXColumn:
    """Fallback: binary op via PyArrow compute."""
    _MAP = {
        "__add__":      pc.add,
        "__sub__":      pc.subtract,
        "__mul__":      pc.multiply,
        "__truediv__":  pc.divide,
    }
    fn       = _MAP[op]
    b_data   = b._data if isinstance(b, MXColumn) else b
    result   = fn(a._data, b_data)
    return MXColumn(result, device=a._device)

## Frame-level helpers

In [ ]:
#| export
def col(frame: MXFrame, name: str, device: DeviceType = "cpu") -> MXColumn:
    """Get a column from `frame` as an `MXColumn`."""
    return MXColumn(frame.to_arrow().column(name), name=name, device=device)


def filter_frame(frame: MXFrame, mask: MXColumn) -> MXFrame:
    """
    Filter `frame` rows by a boolean `mask` MXColumn.

    The mask should contain 1.0 / 0.0 or True / False values.
    PyArrow's zero-copy filter is used under the hood.
    """
    bool_arr = pa.chunked_array([pa.array(mask.to_numpy().astype(bool))])
    return MXFrame(pc.filter(frame.to_arrow(), bool_arr))


def assign_columns(frame: MXFrame, **kwargs: Union[MXColumn, np.ndarray, list]) -> MXFrame:
    """
    Return a new `MXFrame` with extra/replaced columns.

    Examples:
        >>> new_df = assign_columns(df,
        ...     disc_price=col(df, 'price') * (1 - col(df, 'disc')))
    """
    table = frame.to_arrow()
    for name, val in kwargs.items():
        if isinstance(val, MXColumn):
            arr = val.to_arrow()
        elif isinstance(val, np.ndarray):
            arr = pa.chunked_array([pa.array(val)])
        elif isinstance(val, list):
            arr = pa.chunked_array([pa.array(val)])
        else:
            raise TypeError(f"Cannot assign column of type {type(val)}")
        if name in table.column_names:
            table = table.set_column(table.column_names.index(name), name, arr)
        else:
            table = table.append_column(name, arr)
    return MXFrame(table)

## Tests — basic ops

In [ ]:
import numpy as np
from mxframe.core_bridge import MXFrame

# --- Arithmetic ---
c = MXColumn.from_list([1.0, 2.0, 3.0, 4.0], name="v")
assert (c * 2).to_pylist() == [2.0, 4.0, 6.0, 8.0], "mul failed"
assert (c + 1).to_pylist() == [2.0, 3.0, 4.0, 5.0], "add failed"
print("Arithmetic OK")

# --- Comparisons ---
mask = c > 2
assert mask.to_numpy().astype(bool).tolist() == [False, False, True, True], "gt failed"
print("Comparison OK")

# --- Aggregations ---
assert abs(c.sum() - 10.0) < 1e-5, "sum failed"
assert abs(c.mean() - 2.5) < 1e-5, "mean failed"
print("Aggregation OK")

In [ ]:
# --- filter_frame & assign_columns ---
df   = MXFrame({"price": [10.0, 20.0, 30.0], "disc": [0.1, 0.2, 0.0]})
mask = col(df, "price") > 15
filt = filter_frame(df, mask)
assert filt.num_rows == 2, f"filter_frame: expected 2 rows, got {filt.num_rows}"

disc_price = col(df, "price") * (1 - col(df, "disc"))
df2 = assign_columns(df, disc_price=disc_price)
assert "disc_price" in df2.columns, "assign_columns failed"
expected = [9.0, 16.0, 30.0]
actual   = df2["disc_price"].to_pylist()
assert all(abs(a - e) < 1e-4 for a, e in zip(actual, expected)), f"disc_price mismatch: {actual}"
print("filter_frame + assign_columns OK")